# SAIGE — DPO Fine-Tuning (v2)
**Model**: Qwen/Qwen2.5-3B-Instruct  
**Method**: Direct Preference Optimization (DPO) via TRL  
**Dataset**: M1ztyk/SAIGE-right-speech-dpo (85 pairs, prompt-diversified: rs/generic/none conditions)  
**Output**: M1ztyk/SAIGE-dpo-v2

**What changed from v1**: Dataset now stratifies system prompt conditions across three variants per record — RS prompt, generic prompt, and no system prompt. Teaches prompt-independent behavior rather than prompt-activated response.

In [ ]:
# Verify GPU
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(result.stdout.strip())

In [ ]:
# Install dependencies
%pip install -q \
    transformers \
    trl \
    peft \
    accelerate \
    bitsandbytes \
    datasets \
    huggingface_hub

## Authentication
Token needs `write` scope. Get one at https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "M1ztyk/SAIGE-right-speech-dpo",
    data_files="dpo_pairs.jsonl",
    split="train",
)
dataset = dataset.select_columns(["prompt", "chosen", "rejected"])

split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)} pairs | Eval: {len(eval_dataset)} pairs")
print(f"Example prompt keys: {[m['role'] for m in train_dataset[0]['prompt']]}")

## Load Model + Tokenizer (QLoRA)
4-bit NF4 quantization via bitsandbytes — fits comfortably on T4 (16GB).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4 is Turing (CC 7.5) — BF16 requires Ampere+
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded. Params: {model.num_parameters() / 1e9:.2f}B")

## LoRA Configuration

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

## DPO Training

Key hyperparameters — same as v1 (change data, not config, for a clean before/after):
- `beta=0.1` — how far the policy can drift from the reference model
- `lr=5e-7` — DPO is sensitive to LR; lower than SFT
- `epochs=3` — watch eval loss for overfitting at this dataset size
- `ref_model=None` — TRL uses the frozen base (pre-LoRA) as reference automatically

After training, run the 2x2 ablation in the inference notebook:
adapter+RS prompt / adapter+generic prompt / base+RS prompt / base+generic prompt.
The goal: adapter+generic should look close to adapter+RS.

In [ ]:
from trl import DPOTrainer, DPOConfig

OUTPUT_DIR = "./saige-dpo-v2-output"

training_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch size = 16
    learning_rate=5e-7,
    beta=0.1,
    fp16=True,                        # T4 supports FP16, not BF16
    optim="paged_adamw_32bit",
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    max_length=1024,
    max_prompt_length=512,
    report_to="none",
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.train()

## Push to Hub

In [ ]:
OUTPUT_REPO = "M1ztyk/SAIGE-dpo-v2"

trainer.push_to_hub(OUTPUT_REPO)
tokenizer.push_to_hub(OUTPUT_REPO)

print(f"Done. Model at: https://huggingface.co/{OUTPUT_REPO}")